# Sprint 01 — Acquisition des données (raw → bronze)

Première étape du médaillon : charger les **sources brutes** fournies dans la couche
`bronze` (fidèle à la source). C'est le **seul** endroit où le brut est lu ; les
analyses (`01_analyse_exploratoire.ipynb`) lisent ensuite **bronze**.

Sources brutes (`data/raw/`) :
- `telemetry.csv` — relevés capteurs horaires ;
- `releves_incidents.csv` — incidents (données personnelles, anonymisées au stade analyse) ;
- `machine.sql` — référentiel machines & maintenance (dump SQL).

> **Prérequis** : base PostgreSQL démarrée (conteneur `docker/pgdocker`).

## 1. Aperçu des sources brutes

Coup d'œil direct aux CSV bruts (avant chargement) : dimensions, types, premières lignes.

In [1]:
from indusense.data import loaders

telemetry = loaders.load_telemetry()
incidents = loaders.load_incidents()
telemetry.shape, incidents.shape

((134280, 7), (900, 18))

In [2]:
telemetry.head()

,machine_id,timestamp,temperature_c,pressure_bar,voltage_mean_v,rotation_mean_rpm,pieces_produced
0,MACH-01,2025-06-01 00:00:00,45.44,194.302,227.57,1441.7,4
1,MACH-01,2025-06-01 01:00:00,47.87,194.391,227.48,1437.8,4
2,MACH-01,2025-06-01 02:00:00,50.46,195.641,228.68,1484.6,13
3,MACH-01,2025-06-01 03:00:00,48.62,197.737,228.44,1488.9,10
4,MACH-01,2025-06-01 04:00:00,51.09,196.253,227.84,1448.6,6


In [3]:
telemetry.info()
telemetry.isna().sum()

<class 'pandas.DataFrame'>
RangeIndex: 134280 entries, 0 to 134279
Data columns (total 7 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   machine_id         134280 non-null  str    
 1   timestamp          134280 non-null  str    
 2   temperature_c      134280 non-null  float64
 3   pressure_bar       134280 non-null  float64
 4   voltage_mean_v     134280 non-null  float64
 5   rotation_mean_rpm  134280 non-null  float64
 6   pieces_produced    134280 non-null  int64  
dtypes: float64(4), int64(1), str(2)
memory usage: 10.5 MB


machine_id           0
timestamp            0
temperature_c        0
pressure_bar         0
voltage_mean_v       0
rotation_mean_rpm    0
pieces_produced      0
dtype: int64

## 2. Chargement raw → bronze

Équivalent notebook de `uv run indusense-ingest --migrate` : crée le schéma `bronze`,
charge le référentiel (`machine.sql`) puis les CSV (idempotent — tables tronquées avant).

In [4]:
from indusense.data import ingest
from indusense.data.db import get_engine

engine = get_engine()
ingest.setup_bronze(engine, migrate=True)  # bronze + référentiel (machine.sql)
for name in ("telemetry", "incidents"):
    n = ingest.load_csv_to_bronze(engine, ingest.SOURCES[name])
    print(f"{name}: {n} lignes -> bronze.{ingest.SOURCES[name].table}")

Données de référence (machine, maintenance) chargées dans bronze.


telemetry: 134280 lignes -> bronze.telemetry
incidents: 900 lignes -> bronze.incident


## 3. Contrôle de chargement (réconciliation)

Vérifie que `bronze` reflète fidèlement le brut (comptes raw vs bronze).

In [5]:
from indusense.data import loaders
from indusense.data.db import read_bronze

print("Réconciliation raw vs bronze :")
for label, raw_df, table in (
    ("telemetry", loaders.load_telemetry(), "telemetry"),
    ("incidents", loaders.load_incidents(), "incident"),
):
    n_raw, n_bronze = len(raw_df), len(read_bronze(table))
    print(f"  {label:10s} raw={n_raw:>6} bronze={n_bronze:>6}  "
          f"{'OK' if n_raw == n_bronze else 'ECART'}")

print("Référentiel (chargé depuis machine.sql) :")
for table in ("machine", "maintenance"):
    print(f"  bronze.{table}: {len(read_bronze(table))} lignes")

Réconciliation raw vs bronze :


  telemetry  raw=134280 bronze=134280  OK
  incidents  raw=   900 bronze=   900  OK
Référentiel (chargé depuis machine.sql) :
  bronze.machine: 15 lignes
  bronze.maintenance: 115 lignes
